In [ ]:
# SPDX-FileCopyrightText: 2024 Dan J. Bower <dbower@eaps.ethz.ch>
#
# SPDX-License-Identifier: GPL-3.0-or-later

import copy
import logging
from typing import Literal

import numpy as np

from atmodeller import (
    ChemicalSpecies,
    EquilibriumModel,
    Planet,
    PurePhase,
    ReservoirSpecies,
    bulk_silicate_earth_abundances,
    debug_logger,
    earth,
)
from atmodeller.sci_utils import trappist1e as trappist1e_parameters
from atmodeller.solubility import get_solubility_models
from atmodeller.thermodata import IronWustiteBuffer

logger = debug_logger()
logger.setLevel(logging.INFO)

# For more output use DEBUG
# logger.setLevel(logging.DEBUG)

# For no particular reason, use 24 as the random seed
RANDOM_SEED = 24

# TRAPPIST-1e models

This notebook is available at `notebooks/examples/trappist1e.ipynb` and is easiest to obtain by downloading the source code.

## Citation and open access publication

Dan J. Bower, Maggie A. Thompson, Kaustubh Hakim, Meng Tian, and Paolo A. Sossi (2025), Diversity of Low-mass Planet Atmospheres in the C–H–O–N–S–Cl System with Interior Dissolution, Nonideality, and Condensation: Application to TRAPPIST-1e and Sub-Neptunes. The Astrophysical Journal, Volume 995, Number 1, doi: 10.3847/1538-4357/ae1479.

https://iopscience.iop.org/article/10.3847/1538-4357/ae1479

## Initial setup

Parameters

In [ ]:
# In the paper we perform 10,000 simulations
# number_of_realisations = 10000
number_of_realisations = 10

magma_ocean_temperature = 1800
# In the paper we perform simulations at 0.1 and 1.0 melt fraction
mantle_melt_fraction = 1.0

# Venus-like surface temperature
hot_surface_temperature = 740

# Corresponds to the highest temperature at which all condensates can form
medium_surface_temperature = 380

# Corresponds to the equilibrium temperature of Trappist-1e
cool_surface_temperature = 280

# Whether to export the data to Excel and pickle files
WRITE_OUTPUT = False

# There are a few different output formats
output_format: Literal["elements_species", "named_arrays"] = "named_arrays"

# For naming output data
magma_ocean_temp_str: str = f"{magma_ocean_temperature:0.0f}"
hot_temp_str: str = f"{hot_surface_temperature:0.0f}"
medium_temp_str: str = f"{medium_surface_temperature:0.0f}"
cool_temp_str: str = f"{cool_surface_temperature:0.0f}"

Create the species

In [ ]:
H2O_g = ChemicalSpecies.create_gas("H2O")
H2_g = ChemicalSpecies.create_gas("H2")
O2_g = ChemicalSpecies.create_gas("O2")
CO_g = ChemicalSpecies.create_gas("CO")
CO2_g = ChemicalSpecies.create_gas("CO2")
CH4_g = ChemicalSpecies.create_gas("CH4")
N2_g = ChemicalSpecies.create_gas("N2")
H3N_g = ChemicalSpecies.create_gas("H3N")
S2_g = ChemicalSpecies.create_gas("S2")
H2S_g = ChemicalSpecies.create_gas("H2S")
O2S_g = ChemicalSpecies.create_gas("O2S")
OS_g = ChemicalSpecies.create_gas("OS")
Cl2_g = ChemicalSpecies.create_gas("Cl2")
ClH_g = ChemicalSpecies.create_gas("ClH")

# Graphite can be present in high temperature atmospheres
C_s = PurePhase.from_species("C")
# Below are condensates relevant for cooler atmospheres
H2O_l = PurePhase.from_species("H2O", state="l")  # Temperature must be less than 600 K
S_s = PurePhase.from_species("S")  # Temperature must be less than 388.36 K
ClH4N_s = PurePhase.from_species("ClH4N")

# Gas species are the same for all simulations
gas_species = (
    H2_g,
    H2O_g,
    O2_g,
    CO_g,
    CO2_g,
    CH4_g,
    N2_g,
    H3N_g,
    S2_g,
    H2S_g,
    O2S_g,
    OS_g,
    Cl2_g,
    ClH_g,
)

# For high temperature cases the only (potentially) stable condensate is graphite
condensates_graphite_only = (C_s,)

## High temperature atmospheric diversity (gas + C_s)

### Fugacity and mass constraints

In [ ]:
np.random.seed(RANDOM_SEED)
# Log uniform sampling
log10_number_oceans = np.random.uniform(-1, 1, number_of_realisations)
log10_ch_ratios = np.random.uniform(-1, 1, number_of_realisations)
fO2_log10_shifts = np.random.uniform(-5, 5, number_of_realisations)

h_kg = earth.oceans_to_hydrogen_mass(10**log10_number_oceans)
c_kg = h_kg * 10**log10_ch_ratios

fugacity_constraints = {"O2_g": IronWustiteBuffer(fO2_log10_shifts)}

# BSE abundances for Earth
earth_bse = bulk_silicate_earth_abundances()

# Assume the same concentrations of elements in the BSE of Trappist-1e as Earth
trappist1e_bse = copy.deepcopy(earth_bse)
mass_scale_factor = trappist1e_parameters.mantle_mass / earth.mantle_mass

for element, values in trappist1e_bse.items():
    trappist1e_bse[element] = {key: value * mass_scale_factor for key, value in values.items()}  # type: ignore

# Oxygen is determined by the fugacity constraint, so we don't need to include it in the mass
# constraints
mass_constraints = {
    "H": h_kg,
    "C": c_kg,
    "N": trappist1e_bse["N"]["mean"],
    "S": trappist1e_bse["S"]["mean"],
    "Cl": trappist1e_bse["Cl"]["mean"],
}

### High temperature with no magma solubility

In [ ]:
trappist1e_magma_ocean = Planet.from_species(
    gas_species,
    temperature=magma_ocean_temperature,
    planet_mass=trappist1e_parameters.mass,
    surface_radius=trappist1e_parameters.radius,
    mantle_melt_fraction=mantle_melt_fraction,
    condensates=condensates_graphite_only,
)

model_nosol = EquilibriumModel.from_state(
    trappist1e_magma_ocean,
    mass_constraints=mass_constraints,
    activity_constraints=fugacity_constraints,
)

output_nosol = model_nosol.solve_with_default()

output_nosol.solver_stats_to_logger()

# Quick look at the solution
# output_nosol.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_nosol.to_excel(
        f"t1e_{magma_ocean_temp_str}K_no_solubility", output_format=output_format
    )

    # Write the data to a pickle file with dataframes
    output_nosol.to_pickle(
        f"t1e_{magma_ocean_temp_str}K_no_solubility", output_format=output_format
    )

### High temperature with magma solubility

Create the melt species

In [ ]:
solubility_models = get_solubility_models()

# Exclude mass of species in melt from mass balance constraints, as it is negligible compared to
# the mass of the silicate melt itself.
include_in_phase_mass = False

# Melt species
H2O_d = ReservoirSpecies.create_dissolved(
    "H2O",
    solubility=solubility_models["H2O_basalt_dixon95"],
    include_in_phase_mass=include_in_phase_mass,
)
H2_d = ReservoirSpecies.create_dissolved(
    "H2",
    solubility=solubility_models["H2_basalt_hirschmann12"],
    include_in_phase_mass=include_in_phase_mass,
)
CO_d = ReservoirSpecies.create_dissolved(
    "CO",
    solubility=solubility_models["CO_basalt_yoshioka19"],
    include_in_phase_mass=include_in_phase_mass,
)
CO2_d = ReservoirSpecies.create_dissolved(
    "CO2",
    solubility=solubility_models["CO2_basalt_dixon95"],
    include_in_phase_mass=include_in_phase_mass,
)
CH4_d = ReservoirSpecies.create_dissolved(
    "CH4",
    solubility=solubility_models["CH4_basalt_ardia13"],
    include_in_phase_mass=include_in_phase_mass,
)
N2_d = ReservoirSpecies.create_dissolved(
    "N2",
    solubility=solubility_models["N2_basalt_libourel03"],
    include_in_phase_mass=include_in_phase_mass,
)
S2_d = ReservoirSpecies.create_dissolved(
    "S2",
    solubility=solubility_models["S2_basalt_boulliung23"],
    include_in_phase_mass=include_in_phase_mass,
)
Cl2_d = ReservoirSpecies.create_dissolved(
    "Cl2",
    solubility=solubility_models["Cl2_basalt_thomas21"],
    include_in_phase_mass=include_in_phase_mass,
)
melt_species = (H2O_d, H2_d, CO_d, CO2_d, CH4_d, N2_d, S2_d, Cl2_d)

Create the model and solve

In [ ]:
trappist1e_magma_ocean = Planet.from_species(
    gas_species,
    temperature=magma_ocean_temperature,
    planet_mass=trappist1e_parameters.mass,
    surface_radius=trappist1e_parameters.radius,
    mantle_melt_fraction=mantle_melt_fraction,
    # Compared to the case with no magma solubility, the only change is that we now include the
    # melt species in the planet definition
    melt_species=melt_species,
    condensates=condensates_graphite_only,
)

model_withsol = EquilibriumModel.from_state(
    trappist1e_magma_ocean,
    mass_constraints=mass_constraints,
    activity_constraints=fugacity_constraints,
)

output_withsol = model_withsol.solve_with_default()

output_withsol.solver_stats_to_logger()

# Quick look at the solution
# output_withsol.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_withsol.to_excel(
        f"t1e_{magma_ocean_temp_str}K_with_solubility", output_format=output_format
    )

    # Write the data to a pickle file with dataframes
    output_withsol.to_pickle(
        f"t1e_{magma_ocean_temp_str}K_with_solubility", output_format=output_format
    )

## Post-magma ocean (solidified planet)

Based on the high-temperature simulations with magma solubility, we obtain the elemental masses in the atmosphere and surface environment. These values are then imposed as mass constraints on the post-magma ocean (cooler) atmospheres, assuming isochemical re-equilibration.

In [ ]:
withsol_dict = output_withsol.to_dict("elements_species")

mass_constraints = {
    "H": withsol_dict["H"]["gas"]["mass"],
    "S": withsol_dict["S"]["gas"]["mass"],
    "N": withsol_dict["N"]["gas"]["mass"],
    "O": withsol_dict["O"]["gas"]["mass"],
    # For C, we need to add the graphite mass to the gas mass since the assumption is that they
    # remain in equilibrium and are available near the planetary surface for isochemical
    # re-equilibration at lower temperatures.
    "C": withsol_dict["C"]["gas"]["mass"] + withsol_dict["C"]["C_s"]["mass"],
    "Cl": withsol_dict["Cl"]["gas"]["mass"],
}

### Hot Venus-like surface temperature

In [ ]:
trappist1e_hot = Planet.from_species(
    gas_species=gas_species,
    temperature=hot_surface_temperature,
    planet_mass=trappist1e_parameters.mass,
    surface_radius=trappist1e_parameters.radius,
    # Melt fraction is always zero because the planet is solid
    mantle_melt_fraction=0.0,
    condensates=condensates_graphite_only,
)

model_hot = EquilibriumModel.from_state(trappist1e_hot, mass_constraints=mass_constraints)

output_hot = model_hot.solve_with_default()

output_hot.solver_stats_to_logger()

# Quick look at the solution
# output_hot.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_hot.to_excel(f"t1e_{hot_temp_str}K")

    # Write the data to a pickle file with dataframes
    output_hot.to_pickle(f"t1e_{hot_temp_str}K")

### Medium surface temperature with condensates

In [ ]:
# All condensates can form at this temperature (subject to stability constraints)
condensates_temperate = (C_s, H2O_l, S_s, ClH4N_s)

trappist1e_medium = Planet.from_species(
    gas_species=gas_species,
    temperature=medium_surface_temperature,
    planet_mass=trappist1e_parameters.mass,
    surface_radius=trappist1e_parameters.radius,
    # Melt fraction is always zero because the planet is solid
    mantle_melt_fraction=0.0,
    condensates=condensates_temperate,
)

model_temperate = EquilibriumModel.from_state(trappist1e_medium, mass_constraints=mass_constraints)

output_medium = model_temperate.solve_with_default()

output_medium.solver_stats_to_logger()

# Quick look at the solution
# output_medium.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_medium.to_excel(f"t1e_{medium_temp_str}K")

    # Write the data to a pickle file with dataframes
    output_medium.to_pickle(f"t1e_{medium_temp_str}K")

### Equilibrium surface temperature with condensates

In [ ]:
# Since the species are the same, we can re-use the temperate model and take advantage of the fact
# that it has already compiled the solver. To do this we must update the thermodynamic state of the
# model to reflect the cooler temperature.
model_temperate = model_temperate.update_state(temperature=cool_surface_temperature)

# Let's also feed in the previous solution, which should be a reasonable starting point for the
# solver at the cooler temperature.
output_cold = model_temperate.solve(output_medium.solution)

output_cold.solver_stats_to_logger()

# Quick look at the solution
# output_cold.quick_look()

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output_cold.to_excel(f"t1e_{cool_temp_str}K")

    # Write the data to a pickle file with dataframes
    output_cold.to_pickle(f"t1e_{cool_temp_str}K")